[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/attention_mechanism.ipynb)

# The attention mechanism, step by step

This notebook follows the *Attention Mechanism* slides. We go through every piece of the mechanism, first with small numbers you can check by hand, and then inside a real encoder–decoder trained here:

1. **Summaries without looking back**: what is lost when a sentence becomes one fixed-size vector.
2. **A toy translation task** (English → Spanish) where the decoder *needs* to look back.
3. **Seq2seq without attention**: the bottleneck, measured.
4. **Attention by hand**: scores → softmax → weights → context vector.
5. **Three ways to compute the score**: dot product, general (Luong) and additive (Bahdanau).
6. **Seq2seq with attention**: the same model, trained again, now with attention.
7. **Looking at the attention weights**: heatmaps and alignment.
8. **From classic attention to Transformers**: queries, keys, values and $\sqrt{d}$.

### Notation (the same as in the slides)

| symbol | meaning | in the code |
|---|---|---|
| $x^{\langle 1 \rangle}, \dots, x^{\langle T_x \rangle}$ | input sequence (source words) | `x` |
| $y^{\langle 1 \rangle}, \dots, y^{\langle T_y \rangle}$ | output sequence (target words) | `y` |
| $t'$ | input (encoder) position | |
| $t$ | output (decoder) step | |
| $a^{\langle t' \rangle}$ | encoder hidden state at input position $t'$ | `a[:, t_prime]` |
| $s^{\langle t \rangle}$ | decoder hidden state at output step $t$ | `s_t` |
| $e^{\langle t,t' \rangle}$ | score of input position $t'$ for output step $t$ | `e` |
| $\alpha^{\langle t,t' \rangle}$ | attention weight: how much $y^{\langle t \rangle}$ attends to $a^{\langle t' \rangle}$ | `alpha` |
| $c^{\langle t \rangle}$ | context vector used at output step $t$ | `c` |

In [ ]:
import math
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

np.set_printoptions(precision=3, suppress=True)
torch.set_printoptions(precision=3, sci_mode=False)

# The models are small and process one word at a time: the CPU is fast enough
# (and usually faster than a GPU for this). Training takes about 30-60 s per model.
device = torch.device("cpu")

## 1. Summaries without looking back

The exercise from the slides: convert each sentence into a **single fixed-size vector of length 4**, using simple rules. Here are the two example rules:

- **Rule A**: $v_1$ = total ASCII sum, $v_2$ = number of words, $v_3$ = average word length, $v_4$ = max ASCII value.
- **Rule B**: $v_1$ = nouns, $v_2$ = verbs, $v_3$ = adjectives, $v_4$ = total words.

In [ ]:
sentences = [
    "The red ball hit the blue box.",
    "A small robot followed the line.",
    "The doctor treated the patient.",
    "The cat jumped over the fence.",
]

def words(sentence):
    return re.findall(r"[a-z]+", sentence.lower())

def rule_a(sentence):
    w = words(sentence)
    letters = "".join(w)
    return [sum(map(ord, letters)), len(w), round(len(letters) / len(w), 2), max(map(ord, letters))]

# A tiny hand-made part-of-speech dictionary (enough for our sentences)
POS = {"ball": "N", "box": "N", "robot": "N", "line": "N", "doctor": "N", "patient": "N",
       "cat": "N", "fence": "N", "dog": "N", "man": "N",
       "hit": "V", "followed": "V", "treated": "V", "jumped": "V", "chased": "V", "bit": "V",
       "red": "A", "blue": "A", "small": "A"}

def rule_b(sentence):
    tags = [POS.get(w) for w in words(sentence)]
    return [tags.count("N"), tags.count("V"), tags.count("A"), len(tags)]

pd.DataFrame({"sentence": sentences,
              "rule A": [rule_a(s) for s in sentences],
              "rule B": [rule_b(s) for s in sentences]})

**Predict first:** you only receive the vector `[2556, 6, 4.0, 118]` (rule A) or `[2, 1, 0, 6]` (rule B), without the sentence. What is the main subject? Which object was acted upon? Was it about animals, people or objects? Did anything *jump*, *hit* or *follow*?

Now look at what happens with two sentences that mean very different things:

In [ ]:
pairs = [("The cat chased the dog.", "The dog chased the cat."),
         ("The dog bit the man.", "The man bit the dog.")]

for s1, s2 in pairs:
    print(f"{s1:<25s} A={rule_a(s1)}  B={rule_b(s1)}")
    print(f"{s2:<25s} A={rule_a(s2)}  B={rule_b(s2)}")
    print("same vectors?", rule_a(s1) == rule_a(s2) and rule_b(s1) == rule_b(s2), "\n")

**Explain:**

- Why is it hard to reconstruct the meaning from the vector? What information was lost?
- Is the vector big enough? Should we make it *infinite*? (A bag-of-words vector with one entry per word of the vocabulary is huge, and still gives the same vector for both sentences above.)
- How could we keep **all the words available** during decoding, and **selectively look at** some of them when needed?

This is exactly the problem of the classic encoder–decoder, and attention is the answer.

## 2. A toy translation task

We need a task where the decoder must *look back* at specific input words. We generate English noun phrases and translate them to Spanish, like in the slides (*the blue house* → *la casa azul*):

- the **adjective moves after the noun**: *the black cat* → *el gato negro*;
- the **article depends on the gender of the noun**, which comes *later* in the input: *the house* → **la** casa, *the cat* → **el** gato;
- the **adjective agrees with the noun**: *red* → *rojo* / *roja*;
- phrases are joined by *and* → *y*, so we can make sentences as long as we want.

In [ ]:
NOUNS = {"house": ("casa", "f"), "box": ("caja", "f"), "ball": ("pelota", "f"),
         "cat": ("gato", "m"), "dog": ("perro", "m"), "car": ("coche", "m")}
ADJECTIVES = {"red": ("rojo", "roja"), "black": ("negro", "negra"), "white": ("blanco", "blanca"),
              "blue": ("azul", "azul"), "green": ("verde", "verde")}   # (masculine, feminine)

def make_phrase(rng):
    noun = rng.choice(list(NOUNS))
    es_noun, gender = NOUNS[noun]
    article = "la" if gender == "f" else "el"
    if rng.random() < 0.6:                       # 60% of the phrases have an adjective
        adj = rng.choice(list(ADJECTIVES))
        es_adj = ADJECTIVES[adj][gender == "f"]
        return ["the", adj, noun], [article, es_noun, es_adj]
    return ["the", noun], [article, es_noun]

def make_pair(n_phrases, rng):
    src, tgt = [], []
    for i in range(n_phrases):
        if i > 0:
            src.append("and")
            tgt.append("y")
        s, t = make_phrase(rng)
        src += s
        tgt += t
    return src, tgt

rng = random.Random(0)
for n in [1, 1, 2, 4]:
    src, tgt = make_pair(n, rng)
    print(" ".join(src), " -> ", " ".join(tgt))

The words are converted to integer ids. Three special tokens: `<pad>` fills the shorter sentences of a batch, `<sos>` (start of sentence) is the first input of the decoder and `<eos>` (end of sentence) marks the end.

In [ ]:
SPECIAL = ["<pad>", "<sos>", "<eos>"]
PAD, SOS, EOS = 0, 1, 2

src_vocab = SPECIAL + ["the", "and"] + list(ADJECTIVES) + list(NOUNS)
tgt_vocab = SPECIAL + ["el", "la", "y"] + sorted({w for pair in ADJECTIVES.values() for w in pair}) \
            + [es for es, _ in NOUNS.values()]
src_index = {w: i for i, w in enumerate(src_vocab)}
tgt_index = {w: i for i, w in enumerate(tgt_vocab)}

def to_tensors(pairs):
    # Converts a list of (src, tgt) word lists to two padded tensors (batch, length), with <eos> at the end.
    Tx = max(len(s) for s, _ in pairs) + 1
    Ty = max(len(t) for _, t in pairs) + 1
    x = torch.full((len(pairs), Tx), PAD)
    y = torch.full((len(pairs), Ty), PAD)
    for i, (s, t) in enumerate(pairs):
        ids = [src_index[w] for w in s] + [EOS]
        x[i, :len(ids)] = torch.tensor(ids)
        ids = [tgt_index[w] for w in t] + [EOS]
        y[i, :len(ids)] = torch.tensor(ids)
    return x, y

x, y = to_tensors([make_pair(1, rng), make_pair(2, rng)])
print("source vocabulary:", len(src_vocab), "words;  target vocabulary:", len(tgt_vocab), "words")
print("x =\n", x)
print("y =\n", y)

## 3. Seq2seq without attention: the bottleneck

The classic encoder–decoder of the slides:

- the **encoder** (a GRU, a simpler cousin of the LSTM) reads $x^{\langle 1 \rangle}, \dots, x^{\langle T_x \rangle}$ and produces the hidden states $a^{\langle 1 \rangle}, \dots, a^{\langle T_x \rangle}$;
- only the **last** one is passed to the decoder: $c = a^{\langle T_x \rangle} = s^{\langle 0 \rangle}$;
- the **decoder** generates one word at a time. At step $t$ it receives the previous word $y^{\langle t-1 \rangle}$ and its previous state $s^{\langle t-1 \rangle}$, computes $s^{\langle t \rangle}$ and predicts $y^{\langle t \rangle}$.

Everything the decoder knows about the input must fit in the 64 numbers of $a^{\langle T_x \rangle}$.

In [ ]:
EMB_DIM, HID_DIM = 32, 64

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(src_vocab), EMB_DIM, padding_idx=PAD)
        self.rnn = nn.GRU(EMB_DIM, HID_DIM, batch_first=True)

    def forward(self, x):
        # x: (B, Tx) word ids
        a, _ = self.rnn(self.embedding(x))            # a: (B, Tx, H), all the encoder states
        lengths = (x != PAD).sum(dim=1)              # real length of each sentence
        a_last = a[torch.arange(x.size(0)), lengths - 1]   # (B, H), state at the <eos> of each sentence
        return a, a_last


class DecoderNoAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(tgt_vocab), EMB_DIM, padding_idx=PAD)
        self.cell = nn.GRUCell(EMB_DIM, HID_DIM)
        self.out = nn.Linear(HID_DIM, len(tgt_vocab))

    def step(self, y_prev, s_prev, a, mask):
        # One decoding step. a and mask are ignored: this decoder cannot look back.
        s_t = self.cell(self.embedding(y_prev), s_prev)   # s<t> = GRU(y<t-1>, s<t-1>)
        return self.out(s_t), s_t, None                   # logits for y<t>, new state, no attention


class Seq2Seq(nn.Module):
    def __init__(self, decoder):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = decoder

    def forward(self, x, y):
        # Training with "teacher forcing": the decoder input at step t is the CORRECT word y<t-1>.
        a, s = self.encoder(x)                   # s<0> = a<Tx>
        mask = x != PAD                          # True on the real words, False on the padding
        y_prev = torch.full((x.size(0),), SOS)
        logits = []
        for t in range(y.size(1)):
            out, s, _ = self.decoder.step(y_prev, s, a, mask)
            logits.append(out)
            y_prev = y[:, t]
        return torch.stack(logits, dim=1)        # (B, Ty, vocab)

    @torch.no_grad()
    def translate(self, x, max_len=40):
        # Inference: the decoder input at step t is its OWN previous prediction.
        a, s = self.encoder(x)
        mask = x != PAD
        y_prev = torch.full((x.size(0),), SOS)
        outputs, alphas = [], []
        for t in range(max_len):
            out, s, alpha = self.decoder.step(y_prev, s, a, mask)
            y_prev = out.argmax(dim=1)
            outputs.append(y_prev)
            alphas.append(alpha)
        return torch.stack(outputs, dim=1), alphas

Training: sentences with 1 to 6 phrases (3 to 23 words), generated on the fly, 64 per batch. The loss is the cross-entropy of each predicted word, ignoring the padding.

In [ ]:
MAX_PHRASES = 6

def train(model, steps=1500, batch_size=64, seed=0):
    rng = random.Random(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    start = time.time()
    for step in range(1, steps + 1):
        pairs = [make_pair(rng.randint(1, MAX_PHRASES), rng) for _ in range(batch_size)]
        x, y = to_tensors(pairs)
        logits = model(x, y)
        loss = F.cross_entropy(logits.reshape(-1, len(tgt_vocab)), y.reshape(-1), ignore_index=PAD)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if step % 300 == 0:
            print(f"step {step:5d}  loss {loss.item():.4f}  ({time.time() - start:.0f} s)")
    return model

torch.manual_seed(0)
model_none = train(Seq2Seq(DecoderNoAttention()))

We evaluate on 200 new sentences for each number of phrases, with two measures: the fraction of sentences translated **exactly** right, and the fraction of words that are correct in the right position.

In [ ]:
def evaluate(model, n_phrases, n=200, seed=1000):
    rng = random.Random(seed + n_phrases)
    pairs = [make_pair(n_phrases, rng) for _ in range(n)]
    x, _ = to_tensors(pairs)
    out, _ = model.translate(x)
    exact, words_ok, words_total = 0, 0, 0
    for i, (_, tgt) in enumerate(pairs):
        pred = [tgt_vocab[j] for j in out[i].tolist()]
        pred = pred[:pred.index("<eos>")] if "<eos>" in pred else pred
        exact += pred == tgt
        words_ok += sum(p == t for p, t in zip(pred, tgt))
        words_total += len(tgt)
    return exact / n, words_ok / words_total

def accuracy_table(models):
    rows = []
    for name, model in models.items():
        for k in range(1, MAX_PHRASES + 1):
            exact, word = evaluate(model, k)
            rows.append({"model": name, "phrases": k, "exact sentence": exact, "word": word})
    return pd.DataFrame(rows)

def plot_accuracy(table):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
    for ax, col in zip(axes, ["exact sentence", "word"]):
        for (name, d), style in zip(table.groupby("model", sort=False), ["o-", "s--", "^:"]):
            ax.plot(d.phrases, d[col], style, label=name)
        ax.set_xlabel("number of phrases in the sentence")
        ax.set_title(f"{col} accuracy")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("accuracy")
    axes[1].legend()
    plt.show()

def translate_sentence(model, sentence):
    x, _ = to_tensors([(sentence.split(), [])])
    out, alphas = model.translate(x)
    pred = [tgt_vocab[j] for j in out[0].tolist()]
    n = pred.index("<eos>") + 1 if "<eos>" in pred else len(pred)
    return pred[:n], alphas[:n]

**Predict first:** the model was trained on sentences with 1 to 6 phrases. How do you expect the accuracy to change with the length of the sentence?

In [ ]:
table_none = accuracy_table({"no attention": model_none})
plot_accuracy(table_none)

for s in ["the black cat",
          "the blue house and the dog",
          "the blue house and the dog and the red ball and the white car"]:
    print(f"{s}\n   -> {' '.join(translate_sentence(model_none, s)[0])}")

**Explain:** short sentences are fine, but the quality drops quickly with the length. The words come out in the right *pattern* (article, noun, adjective, *y*, ...), but which noun and which adjective? That information did not survive the trip through $a^{\langle T_x \rangle}$. This is the **bottleneck problem**: *"summarize a whole book into one fixed-length sentence and then answer questions from that sentence only."*

The idea of attention: keep **all** the encoder states $a^{\langle 1 \rangle}, \dots, a^{\langle T_x \rangle}$ and, at each decoding step, let the decoder **look back** and pick the ones it needs.

## 4. Attention by hand

At each decoding step $t$:

1. compare the decoder state with each encoder state $a^{\langle t' \rangle}$, giving a **score** $e^{\langle t,t' \rangle}$;
2. transform the scores into **attention weights** $\alpha^{\langle t,t' \rangle}$ with the softmax (positive, sum to 1);
3. compute the **context vector** $c^{\langle t \rangle}$ as the weighted average of the encoder states.

We start with 2-dimensional states, so we can draw them. Imagine the input *the black cat &lt;eos&gt;*.

In [ ]:
input_words = ["the", "black", "cat", "<eos>"]
a = np.array([[ 0.2,  0.9],     # a<1>  "the"
              [ 1.0,  0.3],     # a<2>  "black"
              [ 0.8, -0.6],     # a<3>  "cat"
              [-0.7,  0.4]])    # a<4>  "<eos>"
s = np.array([1.0, -0.2])       # the decoder state that asks the question

def draw_states(a, s, c=None, alpha=None):
    plt.figure(figsize=(5, 5))
    for i, w in enumerate(input_words):
        lw = 1 + (8 * alpha[i] if alpha is not None else 0)
        plt.annotate("", a[i], (0, 0), arrowprops=dict(arrowstyle="-|>", color="tab:blue", lw=lw))
        label = f"a<{i+1}> {w}" + (f"\nα={alpha[i]:.2f}" if alpha is not None else "")
        plt.text(*(a[i] * 1.12), label, color="tab:blue", ha="center", va="center")
    plt.annotate("", s, (0, 0), arrowprops=dict(arrowstyle="-|>", color="tab:red", lw=2))
    plt.text(*(s * 1.15), "s", color="tab:red", fontsize=13)
    if c is not None:
        plt.plot(*c, "*", color="tab:green", markersize=18)
        plt.text(c[0] + 0.07, c[1] - 0.12, "c", color="tab:green", fontsize=13)
    plt.axhline(0, color="gray", lw=0.5)
    plt.axvline(0, color="gray", lw=0.5)
    plt.xlim(-1.3, 1.5)
    plt.ylim(-1.3, 1.5)
    plt.gca().set_aspect("equal")
    plt.show()

draw_states(a, s)

### 4.1 Score: the dot product

The simplest score measures **relevance as similarity**:

$$e^{\langle t,t' \rangle} = s \cdot a^{\langle t' \rangle} = \sum_j s_j\, a_j^{\langle t' \rangle}$$

It is large when the two vectors point in the same direction, close to 0 when they are perpendicular, and negative when they point in opposite directions.

**Predict first:** looking at the figure, which input position gets the highest score? And the lowest?

In [ ]:
e = a @ s          # one dot product per encoder state: (Tx, H) @ (H,) -> (Tx,)

for i, (w, ai, ei) in enumerate(zip(input_words, a, e)):
    print(f"e<{i+1}> = s · a<{i+1}> ({w:>5s}) = {s[0]:.1f}*{ai[0]:4.1f} + {s[1]:.1f}*{ai[1]:4.1f} = {ei:5.2f}")

### 4.2 From scores to weights: the softmax

$$\alpha^{\langle t,t' \rangle} = \frac{\exp(e^{\langle t,t' \rangle})}{\sum_{k=1}^{T_x} \exp(e^{\langle t,k \rangle})}$$

In [ ]:
def softmax(e):
    exp_e = np.exp(e - e.max())      # subtracting the max does not change the result (see below)
    return exp_e / exp_e.sum()

alpha = softmax(e)
for w, ei, ai in zip(input_words, e, alpha):
    print(f"{w:>6s}  e = {ei:6.2f}   exp(e) = {np.exp(ei):5.2f}   alpha = {ai:.3f}")
print("sum of the weights:", alpha.sum())

Why the softmax, and not simply dividing each score by the sum of the scores? Three properties matter:

1. **Scores can be negative**, but weights must be positive. Dividing by the sum gives nonsense.
2. **Adding a constant** to all the scores does not change the weights: only the *differences* between scores matter. That is why we can subtract the maximum before `exp`, to avoid overflow.
3. **Multiplying** the scores by a large number makes the distribution *sharper* (close to choosing a single position); by a small number makes it *flatter* (close to a plain average). Keep this in mind for section 8.

In [ ]:
print("1) divide by the sum:", e / e.sum(), "  <- negative 'weights' and values > 1")
print("   softmax          :", softmax(e))

print("\n2) softmax(e + 100) :", softmax(e + 100))
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(e + 1000) / np.exp(e + 1000).sum()
print("   naive exp(e+1000) :", naive, "  <- overflow")

fig, axes = plt.subplots(1, 4, figsize=(13, 2.6), sharey=True)
for ax, k in zip(axes, [0.1, 1, 3, 10]):
    ax.bar(input_words, softmax(k * e), color="tab:blue")
    ax.set_title(f"softmax({k} × e)")
axes[0].set_ylabel("α")
plt.show()

### 4.3 The context vector

$$c^{\langle t \rangle} = \sum_{t'=1}^{T_x} \alpha^{\langle t,t' \rangle} a^{\langle t' \rangle}, \qquad \sum_{t'=1}^{T_x} \alpha^{\langle t,t' \rangle} = 1$$

Since the weights are positive and sum to 1, $c$ is a **weighted average**: it always lies *inside* the region spanned by the encoder states, pulled towards the ones with more weight. (Arrow thickness = weight.)

In [ ]:
c = alpha @ a      # (Tx,) @ (Tx, H) -> (H,), the same as sum(alpha[i] * a[i])

print("c =", " + ".join(f"{al:.2f}*a<{i+1}>" for i, al in enumerate(alpha)), "=", c)
draw_states(a, s, c, alpha)

**Explain:** move the decoder state `s` (e.g. to `[0, 1]` or `[-1, 0]`) and run the cells again. The encoder states never change, but the context vector does. That is the key difference from the model of section 3: **each output step gets its own context vector**, built from *all* the input positions.

### 4.4 Exercise 1 (from the slides): manual attention computation

Given three encoder states $a^{\langle 1 \rangle}, a^{\langle 2 \rangle}, a^{\langle 3 \rangle}$ and the scores $e^{\langle t,1 \rangle} = 1.0$, $e^{\langle t,2 \rangle} = 2.0$, $e^{\langle t,3 \rangle} = 0.0$:

1. Compute $\alpha^{\langle t,1 \rangle}, \alpha^{\langle t,2 \rangle}, \alpha^{\langle t,3 \rangle}$ using the softmax (use $e \approx 2.718$).
2. Which position gets the highest attention?
3. If the encoder states were one-dimensional, $a^{\langle 1 \rangle} = 1$, $a^{\langle 2 \rangle} = 2$, $a^{\langle 3 \rangle} = 3$, compute $c^{\langle t \rangle}$. Is it closer to 1, 2 or 3? Why?

Solve it on paper first, then check with the next cell.

In [ ]:
e_ex = np.array([1.0, 2.0, 0.0])
a_ex = np.array([1.0, 2.0, 3.0])

alpha_ex = softmax(e_ex)
print("exp(e)  =", np.exp(e_ex), " sum =", np.exp(e_ex).sum())
print("alpha   =", alpha_ex)
print("c       =", " + ".join(f"{al:.3f}*{v:.0f}" for al, v in zip(alpha_ex, a_ex)), "=", alpha_ex @ a_ex)

### 4.5 All the output steps at once: the attention matrix

So far, one decoder state. With $T_y$ decoder states stacked as rows of a matrix $S$ ($T_y \times H$) and the encoder states as rows of $A$ ($T_x \times H$), all the scores are a single matrix product, and the weights are a softmax **along each row**:

$$E = S A^\top \;(T_y \times T_x), \qquad \alpha = \text{softmax}_{\text{rows}}(E), \qquad C = \alpha A \;(T_y \times H)$$

Row $t$ of $\alpha$ says where output $t$ looks; this is the matrix shown as a **heatmap**: rows = output steps, columns = input positions.

In [ ]:
S = np.array([[ 0.1,  1.0],    # three decoder states s<1>, s<2>, s<3>
              [ 1.0, -0.8],
              [ 1.0,  0.5]])

E = S @ a.T                                     # (Ty, Tx)
alpha_matrix = np.exp(E) / np.exp(E).sum(axis=1, keepdims=True)
C = alpha_matrix @ a                            # (Ty, H)

# Sanity check: the same as computing step by step with a loop
for t in range(len(S)):
    assert np.allclose(alpha_matrix[t], softmax(a @ S[t]))
    assert np.allclose(C[t], softmax(a @ S[t]) @ a)

print("each row sums to", alpha_matrix.sum(axis=1))

def heatmap(alpha, x_labels, y_labels, title="", ax=None):
    ax = ax or plt.figure(figsize=(0.7 * len(x_labels) + 1.5, 0.5 * len(y_labels) + 1)).gca()
    ax.imshow(alpha, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for i in range(alpha.shape[0]):
        for j in range(alpha.shape[1]):
            ax.text(j, i, f"{alpha[i, j]:.2f}", ha="center", va="center",
                    color="white" if alpha[i, j] > 0.5 else "black", fontsize=8)
    ax.set_xticks(range(len(x_labels)), x_labels, rotation=45, ha="right")
    ax.set_yticks(range(len(y_labels)), y_labels)
    ax.set_xlabel("input position t'")
    ax.set_ylabel("output step t")
    ax.set_title(title)

heatmap(alpha_matrix, input_words, ["s<1>", "s<2>", "s<3>"], "α = softmax(S Aᵀ)")
plt.show()

## 5. Three ways to compute the score

Everything else stays the same (softmax, weighted average); only the score function $e^{\langle t,t' \rangle} = \text{score}(\text{query}, a^{\langle t' \rangle})$ changes. The *query* is the decoder state that asks *"where should I look?"*.

| name | score | learnable parameters |
|---|---|---|
| dot product (Luong) | $s^\top a^{\langle t' \rangle}$ | none |
| general (Luong) | $s^\top W a^{\langle t' \rangle}$ | $W$ |
| additive (Bahdanau) | $v^\top \tanh(W_s\, s + W_a\, a^{\langle t' \rangle})$ | $W_s$, $W_a$, $v$ |

- The **dot product** requires $s$ and $a$ to have the same size and *live in the same space*.
- The **general** score inserts a learnable matrix: the model learns *which* similarity matters.
- The **additive** score is a small neural network with one hidden layer ($\tanh$) and one output ($v^\top$), applied to each pair (query, $a^{\langle t' \rangle}$).

From now on we work with **batches** in PyTorch: `query` has shape `(B, H)` and `a` has shape `(B, Tx, H)`.

In [ ]:
class DotScore(nn.Module):
    def forward(self, query, a):
        # (B, Tx, H) @ (B, H, 1) -> (B, Tx)
        return torch.bmm(a, query.unsqueeze(2)).squeeze(2)


class GeneralScore(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.W = nn.Linear(hid_dim, hid_dim, bias=False)

    def forward(self, query, a):
        # s^T W a  ==  (W a) · s
        return torch.bmm(self.W(a), query.unsqueeze(2)).squeeze(2)


class AdditiveScore(nn.Module):
    def __init__(self, hid_dim, attn_dim):
        super().__init__()
        self.W_s = nn.Linear(hid_dim, attn_dim, bias=False)
        self.W_a = nn.Linear(hid_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, query, a):
        # W_s s: (B, 1, attn_dim) is added to every W_a a<t'>: (B, Tx, attn_dim)   (broadcasting)
        hidden = torch.tanh(self.W_s(query).unsqueeze(1) + self.W_a(a))
        return self.v(hidden).squeeze(2)          # (B, Tx)


def attend(score_fn, query, a, mask):
    # The whole attention mechanism: scores -> softmax -> weighted average.
    e = score_fn(query, a)                        # (B, Tx)
    e = e.masked_fill(~mask, float("-inf"))       # padding positions: exp(-inf) = 0, so alpha = 0
    alpha = F.softmax(e, dim=1)                   # (B, Tx), each row sums to 1
    c = torch.bmm(alpha.unsqueeze(1), a).squeeze(1)   # (B, 1, Tx) @ (B, Tx, H) -> (B, H)
    return c, alpha

A check with the 2-D example of section 4: the dot score must give the same numbers. Note the **mask**: in a batch, short sentences are padded with `<pad>`, which are not real words and must receive **zero** attention. Here we pretend the second sentence of the batch has only 3 real positions.

In [ ]:
a_t = torch.tensor(a, dtype=torch.float32)
s_t = torch.tensor(s, dtype=torch.float32)
a_batch = torch.stack([a_t, a_t])                  # (B=2, Tx=4, H=2)
s_batch = torch.stack([s_t, s_t])                  # (B=2, H=2)
mask = torch.tensor([[True, True, True, True],
                     [True, True, True, False]])  # last position of sentence 2 is padding

c_b, alpha_b = attend(DotScore(), s_batch, a_batch, mask)
print("alpha (numpy, section 4):", alpha)
print("alpha (torch, batch)    :\n", alpha_b)
print("c:\n", c_b)

torch.manual_seed(0)
for name, fn in [("dot", DotScore()), ("general", GeneralScore(2)), ("additive", AdditiveScore(2, 3))]:
    n_params = sum(p.numel() for p in fn.parameters())
    print(f"{name:>8s} score  parameters: {n_params:2d}   e = {fn(s_batch[:1], a_batch[:1]).detach().numpy().round(2)}")

**Explain:** the *general* and *additive* scores above give different (arbitrary) numbers, because their parameters are random. They only become meaningful after training, together with the encoder and the decoder. Why does the second row of `alpha_b` have a 0 in the last position, and why are the other three weights larger than in the first row?

### Which decoder state is the query: $s^{\langle t-1 \rangle}$ or $s^{\langle t \rangle}$?

The two classic papers place the attention at different moments of the decoding step:

**Bahdanau** (query = *previous* state $s^{\langle t-1 \rangle}$): look first, then update.
1. $c^{\langle t \rangle} = \text{attend}(s^{\langle t-1 \rangle}, a)$
2. $s^{\langle t \rangle} = \text{GRU}([y^{\langle t-1 \rangle}; c^{\langle t \rangle}],\ s^{\langle t-1 \rangle})$ — the context is an *input* of the recurrent cell
3. predict $y^{\langle t \rangle}$ from $[s^{\langle t \rangle}; c^{\langle t \rangle}]$

**Luong** (query = *current* state $s^{\langle t \rangle}$): update first, then look.
1. $s^{\langle t \rangle} = \text{GRU}(y^{\langle t-1 \rangle},\ s^{\langle t-1 \rangle})$
2. $c^{\langle t \rangle} = \text{attend}(s^{\langle t \rangle}, a)$
3. predict $y^{\langle t \rangle}$ from $[s^{\langle t \rangle}; c^{\langle t \rangle}]$

The code follows these steps line by line:

In [ ]:
class DecoderBahdanau(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(tgt_vocab), EMB_DIM, padding_idx=PAD)
        self.score = AdditiveScore(HID_DIM, HID_DIM)
        self.cell = nn.GRUCell(EMB_DIM + HID_DIM, HID_DIM)      # input = [y<t-1>; c<t>]
        self.out = nn.Linear(2 * HID_DIM, len(tgt_vocab))       # input = [s<t>; c<t>]

    def step(self, y_prev, s_prev, a, mask):
        c, alpha = attend(self.score, s_prev, a, mask)                          # 1. query = s<t-1>
        s_t = self.cell(torch.cat([self.embedding(y_prev), c], dim=1), s_prev)  # 2.
        return self.out(torch.cat([s_t, c], dim=1)), s_t, alpha                 # 3.


class DecoderLuong(nn.Module):
    def __init__(self, score):
        super().__init__()
        self.embedding = nn.Embedding(len(tgt_vocab), EMB_DIM, padding_idx=PAD)
        self.score = score
        self.cell = nn.GRUCell(EMB_DIM, HID_DIM)                # input = y<t-1> only
        self.out = nn.Linear(2 * HID_DIM, len(tgt_vocab))       # input = [s<t>; c<t>]

    def step(self, y_prev, s_prev, a, mask):
        s_t = self.cell(self.embedding(y_prev), s_prev)                         # 1.
        c, alpha = attend(self.score, s_t, a, mask)                             # 2. query = s<t>
        return self.out(torch.cat([s_t, c], dim=1)), s_t, alpha                 # 3.

## 6. Seq2seq with attention

Same encoder, same data, same training: only the decoder changes. We train one model with Bahdanau attention and one with Luong dot-product attention.

In [ ]:
torch.manual_seed(0)
model_bahdanau = train(Seq2Seq(DecoderBahdanau()))

torch.manual_seed(0)
model_luong = train(Seq2Seq(DecoderLuong(DotScore())))

**Predict first:** how will the accuracy curves of the two new models compare with the model without attention?

In [ ]:
models = {"no attention": model_none, "Bahdanau (additive)": model_bahdanau, "Luong (dot)": model_luong}
table = accuracy_table(models)
plot_accuracy(table)
table.pivot(index="phrases", columns="model", values="exact sentence")[list(models)]

**Explain:** the model without attention has to *remember* the whole sentence in $a^{\langle T_x \rangle}$. The models with attention only need to remember *where they are* in the translation: the content is always available in $a^{\langle 1 \rangle}, \dots, a^{\langle T_x \rangle}$. The Luong model has almost the same number of parameters as the model without attention (check it below), so the improvement comes from the *architecture*, not from a bigger model.

In [ ]:
for name, m in models.items():
    print(f"{name:>20s}: {sum(p.numel() for p in m.parameters()):6d} parameters")

## 7. Looking at the attention weights

### Exercise 2 (from the slides): interpret an attention heatmap

Imagine an attention matrix where, when predicting the **third** output word, most weight is on the **first** input word, and when predicting the **fourth** output word, most weight is on the **third** input word.

In [ ]:
alpha_ex2 = np.array([[0.10, 0.80, 0.05, 0.05],
                      [0.05, 0.10, 0.80, 0.05],
                      [0.85, 0.05, 0.05, 0.05],
                      [0.05, 0.05, 0.85, 0.05]])
heatmap(alpha_ex2, ["in 1", "in 2", "in 3", "in 4"], ["out 1", "out 2", "out 3", "out 4"], "Exercise 2")
plt.show()

1. What does this suggest about the **alignment** between input and output?
2. How could this be useful in machine translation?

Now the real thing: the attention weights learned by our models. Each row is one output word $y^{\langle t \rangle}$ and shows the weights $\alpha^{\langle t,t' \rangle}$ over the input positions.

### Worked example: *the black cat*

From the slides: when predicting the word for *cat*, the decoder should pay high attention to *cat*, some to *black* and less to *the*. And here there is more: the article **el** depends on the gender of *cat*, which comes **after** *the*.

**Predict first:** where should each of the output words *el*, *gato*, *negro* look?

In [ ]:
def compare_attention(sentence, models):
    fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 0.45 * len(sentence.split()) + 2.5))
    src = sentence.split() + ["<eos>"]
    for ax, (name, model) in zip(axes, models.items()):
        pred, alphas = translate_sentence(model, sentence)
        alpha_matrix = torch.cat(alphas).numpy()      # (Ty, Tx)
        heatmap(alpha_matrix, src, pred, name, ax=ax)
    plt.tight_layout()
    plt.show()

attention_models = {"Bahdanau (additive)": model_bahdanau, "Luong (dot)": model_luong}
compare_attention("the black cat", attention_models)

**Explain** (your numbers may be slightly different, but the patterns should be similar):

- Look at the rows of **gato** and **negro**. The order of the words changes (*black cat* → *gato negro*), so the alignment is **not diagonal**. Attention learns this reordering **by itself**: nobody told the model which words correspond to which.
- Look at the row of **el** in the Bahdanau model. Which input word does it attend to? Why is that the useful word for choosing between *el* and *la*?
- In the Luong model, **el** may look at *&lt;eos&gt;* instead. Remember that $a^{\langle t' \rangle}$ is **not the word** $x^{\langle t' \rangle}$: it is the encoder state after reading $x^{\langle 1 \rangle}, \dots, x^{\langle t' \rangle}$. What does $a^{\langle T_x \rangle}$, the state at *&lt;eos&gt;*, contain? (Hint: it is the context vector of the model without attention.)

A longer sentence:

In [ ]:
compare_attention("the blue house and the dog and the red ball", attention_models)

**Explain:**

- Can you see the **alignment** between the Spanish and the English phrases, phrase by phrase?
- In the Bahdanau model, **azul** may look at *house* rather than at *blue*. Why can $a^{\langle 3 \rangle}$ (*house*) be enough to produce *azul*? (Same reason as above: it is the state after reading *the blue house*.)
- Where does each *y* look? Is *and* always the most useful position to produce *y*?
- The two models do not always look at the same positions, yet both translate correctly. Attention weights show what was **useful for the model**, which is not necessarily what a human would choose. Be careful when using them as an *explanation*.

## 8. From classic attention to Transformers

Look again at the matrix form of section 4.5: $\alpha = \text{softmax}(S A^\top)$, $C = \alpha A$. Transformers give names to the three roles:

| role | classic attention | Transformer |
|---|---|---|
| **query**: who is asking "where should I look?" | decoder state $s^{\langle t \rangle}$ | $Q$ |
| **key**: what is compared with the query | encoder state $a^{\langle t' \rangle}$ | $K$ |
| **value**: what is averaged into the result | encoder state $a^{\langle t' \rangle}$ | $V$ |

In classic attention the keys and the values are the same vectors. Transformers obtain $Q$, $K$ and $V$ with three different learnable matrices, and use the **scaled dot product**:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d}}\right) V$$

### Why divide by $\sqrt{d}$?

If the components of $q$ and $k$ are random with variance 1, the dot product $q \cdot k$ is a sum of $d$ terms, so its standard deviation grows like $\sqrt{d}$. Remember section 4.2: large scores make the softmax **very sharp** (almost one-hot), and then the gradients almost vanish.

In [ ]:
gen = torch.Generator().manual_seed(0)
print(f"{'d':>5s} {'std(q·k)':>9s} {'max α (no scaling)':>19s} {'max α (÷√d)':>12s}")
for d in [4, 16, 64, 256, 1024]:
    q = torch.randn(1000, d, generator=gen)
    k = torch.randn(1000, 10, d, generator=gen)              # 1000 queries, each with 10 keys
    scores = torch.einsum("nd,ntd->nt", q, k)
    max_raw = F.softmax(scores, dim=1).max(dim=1).values.mean()
    max_scaled = F.softmax(scores / math.sqrt(d), dim=1).max(dim=1).values.mean()
    print(f"{d:5d} {scores.std():9.1f} {max_raw:19.2f} {max_scaled:12.2f}")

Without scaling, with large $d$ each query puts almost all its weight on a single key. With $\div\sqrt{d}$ the behaviour does not depend on the dimension.

Our Luong dot-product attention is exactly this formula with $Q$ = the decoder states, $K = V$ = the encoder states (and without the $\sqrt{d}$). Let's check it on the trained model: run the decoder, collect all its states $s^{\langle t \rangle}$ into $Q$, and compute every context vector in **one** matrix operation.

In [ ]:
sentence = "the black cat and the white box"
x_s, _ = to_tensors([(sentence.split(), [])])
with torch.no_grad():
    A, s = model_luong.encoder(x_s)                       # A: (1, Tx, H)
    mask = x_s != PAD
    y_prev = torch.full((1,), SOS)
    states, contexts = [], []
    for t in range(10):                                    # the decoding loop, one step at a time
        out, s, alpha = model_luong.decoder.step(y_prev, s, A, mask)
        states.append(s)
        contexts.append(attend(model_luong.decoder.score, s, A, mask)[0])
        y_prev = out.argmax(dim=1)

    Q = torch.cat(states)                                  # (Ty, H)
    K = V = A[0]                                           # (Tx, H)
    C_matrix = F.softmax(Q @ K.T, dim=1) @ V               # all the steps at once

print("Q:", tuple(Q.shape), " K = V:", tuple(K.shape), " C:", tuple(C_matrix.shape))
print("same context vectors as the loop?", torch.allclose(C_matrix, torch.cat(contexts), atol=1e-5))

In a seq2seq with an RNN, the queries can only be computed one at a time (each $s^{\langle t \rangle}$ depends on $s^{\langle t-1 \rangle}$); the attention itself is just matrix products. Transformers remove the RNN completely and use **self-attention**: every word of a sentence is a query, a key and a value, so each word looks at all the other words of the **same** sentence, all in parallel:

In [ ]:
torch.manual_seed(0)
X = model_luong.encoder.embedding(x_s)[0].detach()        # (T, E): just the word embeddings, no RNN
W_q, W_k, W_v = (nn.Linear(EMB_DIM, 16, bias=False) for _ in range(3))
Q, K, V = W_q(X), W_k(X), W_v(X)
alpha_self = F.softmax(Q @ K.T / math.sqrt(16), dim=1)
Z = alpha_self @ V

print("X:", tuple(X.shape), "-> Z:", tuple(Z.shape), "  attention matrix:", tuple(alpha_self.shape))
words_s = sentence.split() + ["<eos>"]
heatmap(alpha_self.detach().numpy(), words_s, words_s, "self-attention (random, untrained $W_q, W_k, W_v$)")
plt.show()

With random $W_q, W_k, W_v$ the pattern means nothing yet; in a Transformer they are learned, and there are several of them in parallel (**multi-head attention**). That is the subject of the next class.

## Summary

- A fixed-size context vector is a **bottleneck**: performance degrades as the sequences get longer.
- Attention keeps **all** the encoder states and, at each output step, computes **scores** → **weights** (softmax) → **context vector** (weighted average).
- The score can be a dot product, a general bilinear form (Luong) or a small neural network (Bahdanau). Bahdanau uses $s^{\langle t-1 \rangle}$ as the query, Luong uses $s^{\langle t \rangle}$.
- Padding positions are **masked** with $-\infty$ before the softmax.
- The attention weights form a matrix that can be shown as a heatmap and interpreted as a (learned) **alignment**.
- Transformers generalize these ideas: queries, keys and values, the scaled dot product, self-attention and multi-head attention.

## Your turn

1. **General score.** Train `Seq2Seq(DecoderLuong(GeneralScore(HID_DIM)))` and compare its accuracy and its heatmaps with the other models.
2. **Is the vector big enough?** Train the model without attention with `HID_DIM = 256` (change it and re-run the class definitions). Does it close the gap? At what cost in parameters?
3. **Longer than in training.** Evaluate the three models on sentences with 7, 8 and 10 phrases (`evaluate(model, 8)`). Which ones generalize? Why might attention help, and why is it not enough?
4. **Bidirectional encoder.** Make the encoder a bidirectional GRU (`bidirectional=True`, and adapt the sizes). Now $a^{\langle t' \rangle}$ also knows the words *after* $t'$. Does *el* still need to look at the noun? Does *negro* look at *black* more clearly?
5. **No mask.** Remove the `masked_fill` line in `attend` and retrain the Bahdanau model. Translate a short sentence in a batch together with a long one. What happens?
6. Back to section 4: choose `s` so that the attention is almost uniform. And so that it focuses on *cat* only.

In [ ]:
# Your turn